# Notebook 01 - Data Preparation

**Peneliti:** Khoeru Roziqin (24060119120031)  
**Skripsi:** Analisis Sentimen Opini Publik di Platform Media Sosial X Mengenai Program Makan Bergizi Gratis Menggunakan BERT-CNN

Menggabungkan seluruh CSV mentah hasil crawling, melakukan pembersihan dan deduplikasi, kemudian stratified sampling per periode. Output `dataset/final/raw_sampling_mbg.csv` siap untuk anotasi manual pada notebook berikutnya.


## 1. Setup dan Konfigurasi

Notebook dijalankan pada Kaggle Notebook. Direktori output di `/kaggle/working/` mengikuti struktur repositori lokal (`dataset/merge/`, `dataset/final/`, `research/results/`).


In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

assert os.path.exists("/kaggle/working"), "Notebook ini dirancang untuk Kaggle Notebook."
print(f"Pandas {pd.__version__} | NumPy {np.__version__}")


In [ ]:
# -- Path (mirror struktur repositori lokal) -----------------------------------
KAGGLE_INPUT       = "/kaggle/input"
WORK_DIR           = "/kaggle/working"

RAW_DATASET        = os.path.join(KAGGLE_INPUT, "mbg-raw-tweets")
RAW_DATA_MBG_DIR   = os.path.join(RAW_DATASET, "mbg")
RAW_DATA_MAKAN_DIR = os.path.join(RAW_DATASET, "makan_bergizi_gratis")

MERGE_DIR   = os.path.join(WORK_DIR, "dataset", "merge")
FINAL_DIR   = os.path.join(WORK_DIR, "dataset", "final")
RESULTS_DIR = os.path.join(WORK_DIR, "research", "results", "01_data_preparation")
for d in (MERGE_DIR, FINAL_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

MERGED_FULL_PATH       = os.path.join(MERGE_DIR, "merged_mbg_full.csv")
MERGED_FULL_CLEAN_PATH = os.path.join(MERGE_DIR, "merged_mbg_full_clean.csv")
OUTPUT_PATH            = os.path.join(FINAL_DIR, "raw_sampling_mbg.csv")

# -- Rentang waktu (tz-naive) --------------------------------------------------
PIPELINE_START = pd.Timestamp("2025-01-06")   # inklusif
PIPELINE_END   = pd.Timestamp("2026-01-07")   # eksklusif

# -- 12 periode berbasis tanggal 6 (untuk stratified sampling) -----------------
PERIOD_START_STRS = [f"2025-{m:02d}-06" for m in range(1, 13)]
PERIOD_END_STRS   = [f"2025-{m+1:02d}-06" if m < 12 else "2026-01-07"
                     for m in range(1, 13)]
PERIOD_LABELS     = [f"period_{i}" for i in range(1, 13)]

# -- 13 bulan kalender (untuk visualisasi distribusi) --------------------------
MONTH_KEYS = [f"2025-{m:02d}" for m in range(1, 13)] + ["2026-01"]
MONTH_LABELS = [
    "Jan 2025", "Feb 2025", "Mar 2025", "Apr 2025",
    "Mei 2025", "Jun 2025", "Jul 2025", "Agu 2025",
    "Sep 2025", "Okt 2025", "Nov 2025", "Des 2025", "Jan 2026",
]
MONTH_KEY_TO_LABEL = dict(zip(MONTH_KEYS, MONTH_LABELS))

# -- Keyword filter ------------------------------------------------------------
KEYWORDS_MAKAN = ["makan bergizi gratis", "makan bergizi",
                  "makan siang gratis",   "program makan gratis"]
KEYWORDS_MBG   = ["mbg"]

# -- Spam keyword filter -------------------------------------------------------
SPAM_KEYWORDS = [
    "slot", "gacor", "zeus", "pragmatic", "maxwin", "wd", "depo",
    "pola gacor", "rtp", "judol", "link daftar", "situs",
    "shopee", "tokopedia", "lazada", "blibli", "cek keranjang",
    "spaylater", "dana kaget", "giveaway", "racun", "link di bio",
    "affiliate", "open joki", "joki tugas", "jasjok", "convert pulsa",
    "mutualan", "follback", "biro jodoh",
    "open bo", "vcs", "video syur", "museum", "boke", "desah", "colmek",
]
SPAM_REGEX_PATTERN = "|".join(SPAM_KEYWORDS)

# -- Parameter sampling --------------------------------------------------------
SAMPLE_PER_PERIOD = 570   # 500 target + 70 buffer
MIN_WORD_COUNT    = 3

# -- Kolom yang dipertahankan --------------------------------------------------
COLS_KEEP = [
    "id_str", "conversation_id_str", "created_at",
    "full_text", "username", "user_id_str",
    "favorite_count", "retweet_count", "reply_count", "quote_count",
    "lang", "location", "source_keyword",
]

print(f"Output dir     : {WORK_DIR}")
print(f"Sampel/periode : {SAMPLE_PER_PERIOD}")
print(f"Periode        : {len(PERIOD_LABELS)} (basis tanggal 6)")
print(f"Bulan kalender : {len(MONTH_LABELS)} (Jan 2025 - Jan 2026)")


## 2. Fungsi Pipeline

Fungsi modular untuk load, standardisasi, normalisasi timestamp, filter, deduplikasi, dan pemetaan periode.


In [ ]:
def normalize_created_at(df):
    """Normalisasi created_at ke tz-naive UTC. Menangani ISO 8601 dan format Twitter."""
    n_before = len(df)
    df = df.copy()
    df["created_at"] = pd.to_datetime(
        df["created_at"], errors="coerce", format="mixed", utc=True
    )
    df = df.dropna(subset=["created_at"])
    n_dropped = n_before - len(df)
    if n_dropped > 0:
        print(f"  {n_dropped:,} baris dibuang (created_at tidak valid)")
    df["created_at"] = df["created_at"].dt.tz_localize(None)
    return df


def load_and_merge_csv(folder_path):
    """Baca semua CSV pada folder dan gabungkan."""
    all_files = glob.glob(os.path.join(folder_path, "*.csv"))
    if not all_files:
        print(f"  Tidak ada file CSV di: {folder_path}")
        return pd.DataFrame()
    df_list = []
    for fp in all_files:
        try:
            df_raw = pd.read_csv(fp, on_bad_lines="skip", lineterminator="\n",
                                 encoding_errors="ignore", low_memory=False)
            df_list.append(df_raw)
        except Exception as e:
            print(f"  Gagal membaca {os.path.basename(fp)}: {e}")
    return pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()


def standardize_columns(df):
    """Rename kolom dan tangani anomali kolom username / id_str dari raw CSV."""
    df = df.copy()
    df.rename(columns={"date": "created_at", "text": "full_text"},
              inplace=True, errors="ignore")

    # Anomali kolom username (trailing CR atau karakter separator dari CSV raw)
    anomaly_cols = ["username", "username\r", "username;;;;;;;\r"]
    if "username" not in df.columns:
        df["username"] = pd.NA
    for col in anomaly_cols:
        if col in df.columns and col != "username":
            df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)
            df["username"] = df["username"].replace(r"^\s*$", pd.NA, regex=True)
            df["username"] = df["username"].combine_first(df[col])
    df.drop(columns=[c for c in anomaly_cols if c != "username"],
            inplace=True, errors="ignore")

    # id_str: kadang terbaca sebagai float, hapus '.0' di akhir saja
    if "id_str" in df.columns:
        df["id_str"] = df["id_str"].astype(str)
        df["id_str"] = df["id_str"].replace(r"\.0$", "", regex=True)
        df.loc[df["id_str"].isin(["nan", "<NA>"]), "id_str"] = ""
    else:
        print("  Kolom id_str tidak ditemukan.")
        df["id_str"] = ""
    return df


def filter_by_required_keywords(df, required_keywords):
    """Filter baris yang mengandung minimal satu keyword."""
    pattern = "|".join(required_keywords)
    mask    = df["full_text"].astype(str).str.contains(pattern, case=False, na=False)
    return df[mask].copy()


def filter_by_date(df, start_ts, end_ts):
    """Filter created_at dalam rentang [start, end)."""
    start = pd.Timestamp(start_ts)
    end   = pd.Timestamp(end_ts)
    if start.tzinfo is not None: start = start.tz_localize(None)
    if end.tzinfo   is not None: end   = end.tz_localize(None)
    return df[(df["created_at"] >= start) & (df["created_at"] < end)].copy()


def remove_id_duplicates(df):
    """Hapus duplikat berdasarkan id_str."""
    df = df[df["id_str"] != ""].copy()
    df.drop_duplicates(subset=["id_str"], keep="first", inplace=True)
    return df


def remove_advanced_duplicates(df):
    """Hapus duplikat teks setelah menormalisasi URL, mention, hashtag, tanda baca."""
    temp = df["full_text"].astype(str).str.lower()
    temp = temp.str.replace(r"http\S+|www\S+|https\S+", "", regex=True)
    temp = temp.str.replace(r"@\w+",    "", regex=True)
    temp = temp.str.replace(r"#\w+",    "", regex=True)
    temp = temp.str.replace(r"[^\w\s]", "", regex=True)
    temp = temp.str.replace(r"\s+", " ", regex=True).str.strip()
    df   = df.copy()
    df["_temp_dedup"] = temp
    df = df[df["_temp_dedup"] != ""]
    df.drop_duplicates(subset=["_temp_dedup"], keep="first", inplace=True)
    df.drop(columns=["_temp_dedup"], inplace=True)
    return df


def remove_spam(df):
    """Hapus baris yang mengandung kata kunci spam."""
    is_spam = df["full_text"].astype(str).str.contains(
        SPAM_REGEX_PATTERN, case=False, na=False)
    return df[~is_spam].copy()


def assign_period(created_at_series):
    """Petakan timestamp ke period_1..period_12 berbasis tanggal 6."""
    result = pd.Series(pd.NA, index=created_at_series.index, dtype="object")
    for lbl, ps_str, pe_str in zip(PERIOD_LABELS, PERIOD_START_STRS, PERIOD_END_STRS):
        ps   = pd.Timestamp(ps_str)
        pe   = pd.Timestamp(pe_str)
        mask = (created_at_series >= ps) & (created_at_series < pe)
        result[mask] = lbl
    return result


def get_month_key(created_at_series):
    """Ekstrak bulan kalender sebagai string 'YYYY-MM'."""
    return created_at_series.dt.strftime("%Y-%m")


def print_summary_by_keyword(df, label="Dataset"):
    """Cetak total dan breakdown per source_keyword."""
    print(f"  Total {label:<28}: {len(df):>8,} tweet")
    if "source_keyword" in df.columns:
        col = df["source_keyword"]
        if isinstance(col, pd.DataFrame):
            col = col.iloc[:, 0]
        for kw, cnt in col.value_counts().items():
            print(f"    - [{kw:<26}]: {cnt:>8,} tweet")


def process_pipeline(folder_path, source_label, required_keywords):
    """Jalankan pipeline lengkap per kelompok keyword."""
    print(f"\n{'-'*62}")
    print(f" Processing: {source_label}")
    print(f"{'-'*62}")

    df = load_and_merge_csv(folder_path)
    if df.empty: return df
    n0 = len(df); print(f"  [1] Raw CSV loaded           : {n0:>8,} baris")

    df = standardize_columns(df)
    if "full_text" not in df.columns or "created_at" not in df.columns:
        print("  Kolom full_text/created_at tidak ditemukan.")
        return pd.DataFrame()

    df = normalize_created_at(df)
    if df.empty:
        print("  Semua baris dibuang setelah normalisasi.")
        return pd.DataFrame()
    n1 = len(df); print(f"  [2] Setelah normalisasi ts   : {n1:>8,} baris  (-{n0-n1:,})")

    df = filter_by_required_keywords(df, required_keywords)
    n2 = len(df); print(f"  [3] Setelah filter keyword   : {n2:>8,} baris  (-{n1-n2:,})")

    df = filter_by_date(df, PIPELINE_START, PIPELINE_END)
    n3 = len(df); print(f"  [4] Setelah filter tanggal   : {n3:>8,} baris  (-{n2-n3:,})")

    df = remove_id_duplicates(df)
    n4 = len(df); print(f"  [5] Dedup by ID              : {n4:>8,} baris  (-{n3-n4:,})")

    df = remove_advanced_duplicates(df)
    n5 = len(df); print(f"  [6] Dedup by teks            : {n5:>8,} baris  (-{n4-n5:,})")

    df = remove_spam(df)
    n6 = len(df); print(f"  [7] Setelah hapus spam       : {n6:>8,} baris  (-{n5-n6:,})")

    df = df.sort_values(by="created_at", ascending=True).reset_index(drop=True)
    df["source_keyword"] = source_label
    print(f"  FINAL [{source_label:<24}]: {len(df):>8,} baris")
    return df


print("Fungsi pipeline siap.")


## 3. Eksekusi Pipeline, Merge, dan Pre-filter

Pipeline dijalankan per kelompok keyword, digabungkan lintas-dataset, dan diinspeksi sebelum pre-filter berdasarkan panjang teks.


In [ ]:
# -- Eksekusi per kelompok keyword ---------------------------------------------
df_makan = process_pipeline(RAW_DATA_MAKAN_DIR, "makan_bergizi_gratis", KEYWORDS_MAKAN)
df_mbg   = process_pipeline(RAW_DATA_MBG_DIR,   "mbg",                  KEYWORDS_MBG)

# -- Merge dan dedup lintas-dataset --------------------------------------------
print(f"\n{'-'*62}")
print(" Penggabungan Lintas-Dataset")
print(f"{'-'*62}")

df_full = pd.concat([df_makan, df_mbg], axis=0, ignore_index=True)
df_full = df_full.loc[:, ~df_full.columns.duplicated()]

n_concat = len(df_full)
print(f"  Setelah concat             : {n_concat:>8,} baris")
print_summary_by_keyword(df_full, "setelah concat")

dup_id = df_full.duplicated(subset=["id_str"]).sum()
if dup_id > 0:
    df_full.drop_duplicates(subset=["id_str"], keep="first", inplace=True)
print(f"  Dedup ID lintas-dataset    : {len(df_full):>8,} baris  (-{dup_id:,})")

dup_txt = df_full.duplicated(subset=["full_text"]).sum()
if dup_txt > 0:
    df_full.drop_duplicates(subset=["full_text"], keep="first", inplace=True)
print(f"  Dedup teks lintas-dataset  : {len(df_full):>8,} baris  (-{dup_txt:,})")

df_full = df_full.sort_values(by="created_at", ascending=True).reset_index(drop=True)
print(f"\n  MERGED FULL FINAL          : {len(df_full):>8,} baris")
print_summary_by_keyword(df_full, "merged full")

if not df_full.empty:
    df_full.to_csv(MERGED_FULL_PATH, index=False)
    print(f"\n  Saved: {MERGED_FULL_PATH}")

# -- Inspeksi dataset gabungan -------------------------------------------------
print(f"\n{'='*62}")
print(" INSPEKSI DATASET GABUNGAN")
print(f"{'='*62}")
print(f"Shape  : {df_full.shape[0]:,} baris x {df_full.shape[1]} kolom")
print(f"Kolom  : {list(df_full.columns)}")
print(f"\nMissing values per kolom:")
print(df_full.isnull().sum())
print(f"\nRentang waktu:")
print(f"  Terlama : {df_full['created_at'].min()}")
print(f"  Terbaru : {df_full['created_at'].max()}")
print(f"\nDistribusi berdasarkan keyword:")
print_summary_by_keyword(df_full, "dataset gabungan")
print(f"\nPreview 3 baris pertama:")
display(df_full[["id_str","created_at","full_text","source_keyword"]].head(3))
print(f"\nPreview 3 baris terakhir:")
display(df_full[["id_str","created_at","full_text","source_keyword"]].tail(3))

# -- Pre-filter (null dan panjang minimum) -------------------------------------
print(f"\n{'-'*62}")
print(" Pre-filter untuk Sampling")
print(f"{'-'*62}")

cols_needed = list(dict.fromkeys(
    [c for c in COLS_KEEP if c in df_full.columns] + ["source_keyword"]
))
df = df_full[[c for c in cols_needed if c in df_full.columns]].copy()
df = df.loc[:, ~df.columns.duplicated()]

n_pf0 = len(df)
print(f"\n  [0] Data masuk pre-filter      : {n_pf0:>8,} tweet")

df = df.dropna(subset=["full_text"])
df["full_text"] = df["full_text"].astype(str).str.strip()
df = df[df["full_text"] != ""]
n_pf1 = len(df)
print(f"  [1] Setelah filter null/kosong : {n_pf1:>8,} tweet  (-{n_pf0-n_pf1:,})")

df["word_count"] = df["full_text"].str.split().str.len()
df = df[df["word_count"] > MIN_WORD_COUNT]
n_pf2 = len(df)
print(f"  [2] Setelah filter panjang >{MIN_WORD_COUNT}  : {n_pf2:>8,} tweet  (-{n_pf1-n_pf2:,})")

df = df.sort_values(by="created_at", ascending=True).reset_index(drop=True)
print(f"\n  Siap sampling: {len(df):,} tweet")
print_summary_by_keyword(df, "siap sampling")

if not df.empty:
    df.to_csv(MERGED_FULL_CLEAN_PATH, index=False)
    print(f"\n  Saved: {MERGED_FULL_CLEAN_PATH}")


## 4. Visualisasi Tren Dataset Gabungan

Tren volume tweet harian dan bulanan untuk melihat pola temporal.


In [ ]:
def plot_timeline(df_list, labels, title, save_path=None):
    fig, ax = plt.subplots(figsize=(15, 5))
    colors  = ["#534AB7", "#E24B4A", "#1D9E75"]
    for i, (df_plot, label) in enumerate(zip(df_list, labels)):
        if df_plot.empty: continue
        daily = df_plot.set_index("created_at").resample("D").size()
        ax.plot(daily.index, daily.values, label=label,
                color=colors[i % len(colors)], alpha=0.85, linewidth=1.4)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b-%y"))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=14))
    ax.set_title(title, fontsize=13, pad=12)
    ax.set_xlabel("Tanggal", fontsize=11)
    ax.set_ylabel("Jumlah Tweet", fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.legend(fontsize=10); ax.grid(True, alpha=0.25)
    plt.xticks(rotation=45); sns.despine(); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_monthly(df_list, labels, title, save_path=None):
    combined = pd.DataFrame()
    for df_plot, label in zip(df_list, labels):
        if df_plot.empty: continue
        monthly = df_plot.set_index("created_at").resample("ME").size()
        monthly.index = monthly.index.strftime("%b %y")
        combined[label] = monthly
    if combined.empty:
        print("Tidak ada data."); return
    colors = ["#534AB7", "#E24B4A"]
    ax = combined.plot(kind="bar", figsize=(13, 5), color=colors[:len(labels)],
                       width=0.8, alpha=0.9, edgecolor="none")
    ax.set_title(title, fontsize=13, pad=12)
    ax.set_xlabel("Bulan", fontsize=11)
    ax.set_ylabel("Jumlah Tweet", fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    plt.xticks(rotation=0, fontsize=9); ax.grid(axis="y", alpha=0.25)
    for p in ax.patches:
        h = p.get_height()
        if pd.notnull(h) and h > 0:
            ax.annotate(f"{int(h):,}", (p.get_x()+p.get_width()/2, h),
                        ha="center", va="bottom", xytext=(0,4),
                        textcoords="offset points", fontsize=7)
    ax.legend(fontsize=10); sns.despine(); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
plot_timeline([df_full], ["Merged Full Dataset"],
              "Volume Tweet Harian - Dataset MBG Gabungan",
              f"{RESULTS_DIR}/trend_harian_full.png")


In [ ]:
plot_monthly([df_full], ["Merged Full Dataset"],
             "Volume Tweet Bulanan - Dataset MBG Gabungan",
             f"{RESULTS_DIR}/trend_bulanan_full.png")


In [ ]:
plot_timeline([df_makan, df_mbg], ["Makan Bergizi Gratis", "MBG"],
              "Volume Tweet Harian - Per Keyword",
              f"{RESULTS_DIR}/trend_harian_per_keyword.png")


In [ ]:
plot_monthly([df_makan, df_mbg], ["Makan Bergizi Gratis", "MBG"],
             "Volume Tweet Bulanan - Per Keyword",
             f"{RESULTS_DIR}/trend_bulanan_per_keyword.png")


## 5. Assign Periode

Setiap tweet dipetakan ke salah satu dari 12 periode (basis tanggal 6). Kolom `month_key` juga ditambahkan untuk visualisasi berbasis bulan kalender.


In [ ]:
df["period"]    = assign_period(df["created_at"])
df["month_key"] = get_month_key(df["created_at"])

n_unassigned = df["period"].isna().sum()
if n_unassigned > 0:
    print(f"{n_unassigned:,} tweet tidak terassign (di luar rentang)")
df = df.dropna(subset=["period"])

print(f"Assign selesai: {len(df):,} tweet")
print(f"  Period : {df['period'].nunique()} unik (period_1 - period_12)")
print(f"  Bulan  : {df['month_key'].nunique()} unik (bulan kalender)")

# -- Distribusi per period (12 periode) ----------------------------------------
dist_avail_period = (
    df.groupby("period", observed=True).size()
    .reindex(PERIOD_LABELS, fill_value=0).reset_index()
)
dist_avail_period.columns   = ["period", "n_available"]
dist_avail_period["is_low"] = dist_avail_period["n_available"] < SAMPLE_PER_PERIOD

print(f"\n=== DISTRIBUSI DATA TERSEDIA PER PERIODE (12 periode) ===")
print(f"{'Periode':<12} {'n_available':>12}  Status")
print("-" * 40)
for _, row in dist_avail_period.iterrows():
    flag = "Low-volume" if row["is_low"] else "Normal"
    print(f"{row['period']:<12} {row['n_available']:>12,}  {flag}")
print("-" * 40)
print(f"  {'TOTAL':<11} {dist_avail_period['n_available'].sum():>12,}")
print(f"\nPeriode low-volume (<{SAMPLE_PER_PERIOD}): {dist_avail_period['is_low'].sum()} periode")

# -- Distribusi per bulan kalender (13 bulan) ----------------------------------
dist_avail_month = (
    df.groupby("month_key", observed=True).size()
    .reindex(MONTH_KEYS, fill_value=0).reset_index()
)
dist_avail_month.columns        = ["month_key", "n_available"]
dist_avail_month["month_label"] = dist_avail_month["month_key"].map(MONTH_KEY_TO_LABEL)

print(f"\n=== DISTRIBUSI DATA TERSEDIA PER BULAN KALENDER (13 bulan) ===")
print(f"{'Bulan':<18} {'n_available':>12}")
print("-" * 33)
for _, row in dist_avail_month.iterrows():
    print(f"{row['month_label']:<18} {row['n_available']:>12,}")
print("-" * 33)
print(f"  {'TOTAL':<17} {dist_avail_month['n_available'].sum():>12,}")

# -- Distribusi per keyword per periode ----------------------------------------
print(f"\nDistribusi per keyword per periode:")
dist_kw = (
    df.groupby(["period","source_keyword"], observed=True).size()
    .unstack(fill_value=0).reindex(PERIOD_LABELS, fill_value=0)
)
print(dist_kw.to_string())


## 6. Stratified Sampling

Random sampling `SAMPLE_PER_PERIOD` tweet per periode. Periode dengan jumlah data < target menggunakan sensus parsial (seluruh data periode diambil).


In [ ]:
sampled_frames = []
sampling_log   = []

for period_lbl in PERIOD_LABELS:
    df_period = df[df["period"] == period_lbl]
    n_avail   = len(df_period)

    if n_avail == 0:
        sampling_log.append({
            "period": period_lbl, "n_available": 0, "n_sampled": 0,
            "sampling_rate": 0.0, "is_low_volume": True,
            "note": "Tidak ada data",
        })
        continue

    if n_avail >= SAMPLE_PER_PERIOD:
        sampled    = df_period.sample(n=SAMPLE_PER_PERIOD, random_state=RANDOM_SEED)
        n_sampled  = SAMPLE_PER_PERIOD
        is_low_vol = False
        note       = f"Random sample ({SAMPLE_PER_PERIOD:,} dari {n_avail:,})"
    else:
        sampled    = df_period.copy()
        n_sampled  = n_avail
        is_low_vol = True
        note       = f"Sensus parsial ({n_avail:,} < {SAMPLE_PER_PERIOD:,})"

    sampled_frames.append(sampled)
    sampling_log.append({
        "period"       : period_lbl,
        "n_available"  : n_avail,
        "n_sampled"    : n_sampled,
        "sampling_rate": round(n_sampled / n_avail * 100, 2),
        "is_low_volume": is_low_vol,
        "note"         : note,
    })

df_sampled = pd.concat(sampled_frames, ignore_index=True)
df_sampled = df_sampled.sort_values(
    by="created_at", ascending=True).reset_index(drop=True)
df_log = pd.DataFrame(sampling_log)

print("=== LAPORAN SAMPLING PER PERIODE ===")
print(f"{'Periode':<12} {'Tersedia':>10} {'Disampling':>12} {'Rate':>7}  Keterangan")
print("-" * 75)
for _, row in df_log.iterrows():
    print(f"{row['period']:<12} {row['n_available']:>10,} "
          f"{row['n_sampled']:>12,} {row['sampling_rate']:>6.1f}%  {row['note']}")
print("-" * 75)
print(f"{'TOTAL':<12} {df_log['n_available'].sum():>10,} "
      f"{df_log['n_sampled'].sum():>12,}")
print(f"\nTotal sampel: {len(df_sampled):,} tweet")
print_summary_by_keyword(df_sampled, "hasil sampling")


## 7. Distribusi Hasil Sampling

Distribusi hasil sampling divisualisasikan dalam dua sudut pandang: berdasarkan periode sampling (12 bar) dan berdasarkan bulan kalender aktual `created_at` (13 bar).


In [ ]:
# -- Distribusi hasil sampling by period (12 periode) --------------------------
dist_samp_period = df_log[["period","n_sampled","is_low_volume"]].copy()

print("=== DISTRIBUSI HASIL SAMPLING PER PERIODE (12 periode) ===")
print(f"{'Periode':<12} {'n_sampled':>12}  Status")
print("-" * 40)
for _, row in dist_samp_period.iterrows():
    flag = "Sensus parsial" if row["is_low_volume"] else "Full sample"
    print(f"{row['period']:<12} {row['n_sampled']:>12,}  {flag}")
print("-" * 40)
print(f"  {'TOTAL':<11} {dist_samp_period['n_sampled'].sum():>12,}")

# -- Chart by Period -----------------------------------------------------------
bar_colors_p = ["#E24B4A" if lv else "#1D9E75"
                for lv in dist_samp_period["is_low_volume"]]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(dist_samp_period["period"], dist_samp_period["n_sampled"],
              color=bar_colors_p, edgecolor="none", width=0.65)
for bar, val in zip(bars, dist_samp_period["n_sampled"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
            f"{val:,}", ha="center", va="bottom", fontsize=8, color="#444441")
ax.axhline(y=SAMPLE_PER_PERIOD, color="#EF9F27", linestyle="--", linewidth=1.5)
ax.set_xlabel("Periode Sampling", fontsize=11)
ax.set_ylabel("Jumlah Tweet Disampling", fontsize=11)
ax.set_title(f"Distribusi Hasil Sampling per Periode (12 Periode, Basis Tanggal 6)",
             fontsize=12, pad=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.tick_params(axis="x", rotation=45)
ax.legend(handles=[
    Patch(facecolor="#1D9E75", label=f"Full sample ({SAMPLE_PER_PERIOD})"),
    Patch(facecolor="#E24B4A", label="Sensus parsial (low-volume)"),
    plt.Line2D([0],[0], color="#EF9F27", linestyle="--",
               label=f"Target {SAMPLE_PER_PERIOD}/periode"),
], fontsize=9)
sns.despine(); plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/distribusi_sampling_per_periode.png",
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# -- Distribusi hasil sampling by month (13 bulan) -----------------------------
df_sampled["month_key"] = get_month_key(df_sampled["created_at"])
dist_samp_month = (
    df_sampled.groupby("month_key", observed=True).size()
    .reindex(MONTH_KEYS, fill_value=0).reset_index()
)
dist_samp_month.columns        = ["month_key", "n_sampled"]
dist_samp_month["month_label"] = dist_samp_month["month_key"].map(MONTH_KEY_TO_LABEL)

print("=== DISTRIBUSI HASIL SAMPLING PER BULAN KALENDER (13 bulan) ===")
print(f"{'Bulan':<18} {'n_sampled':>12}")
print("-" * 33)
for _, row in dist_samp_month.iterrows():
    print(f"{row['month_label']:<18} {row['n_sampled']:>12,}")
print("-" * 33)
print(f"  {'TOTAL':<17} {dist_samp_month['n_sampled'].sum():>12,}")

# -- Chart by Month ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(dist_samp_month["month_label"], dist_samp_month["n_sampled"],
              color="#534AB7", edgecolor="none", width=0.65)
for bar, val in zip(bars, dist_samp_month["n_sampled"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
            f"{val:,}", ha="center", va="bottom", fontsize=8, color="#444441")
ax.set_xlabel("Bulan Kalender", fontsize=11)
ax.set_ylabel("Jumlah Tweet Disampling", fontsize=11)
ax.set_title("Distribusi Hasil Sampling per Bulan Kalender (13 Bulan, Jan 2025 - Jan 2026)",
             fontsize=12, pad=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.tick_params(axis="x", rotation=45)
sns.despine(); plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/distribusi_sampling_per_bulan.png",
            dpi=150, bbox_inches="tight")
plt.show()

# -- Statistik ringkasan -------------------------------------------------------
print("\n=== STATISTIK RINGKASAN ===")
total_avail = df_log["n_available"].sum()
total_samp  = df_log["n_sampled"].sum()
print(f"Total data siap sampling  : {total_avail:,}")
print(f"Total sampel              : {total_samp:,}")
print(f"Periode normal            : {(~df_log['is_low_volume']).sum()} periode")
print(f"Periode low-volume        : {df_log['is_low_volume'].sum()} periode")
print(f"Overall sampling rate     : {total_samp/total_avail*100:.2f}%")

stats = df_sampled["word_count"].describe()
print(f"\nPanjang tweet (kata):")
print(f"  Mean   : {stats['mean']:.2f}")
print(f"  Median : {stats['50%']:.0f}")
print(f"  Min    : {stats['min']:.0f}")
print(f"  Max    : {stats['max']:.0f}")


## 8. Export Output

Sampel disimpan sebagai `raw_sampling_mbg.csv` dengan tiga kolom anotasi kosong yang akan diisi pada notebook berikutnya.


In [ ]:
output_cols = [
    "id_str", "conversation_id_str", "created_at", "period",
    "full_text", "word_count", "username", "user_id_str",
    "favorite_count", "retweet_count", "reply_count", "quote_count",
    "lang", "location", "source_keyword",
]
output_cols_exist = [c for c in output_cols if c in df_sampled.columns]
df_final = df_sampled[output_cols_exist].copy()

# Kolom anotasi (diisi pada NB02)
df_final["label"]           = ""   # positive / negative / neutral
df_final["label_roberta"]   = ""   # prediksi IndoRoBERTa
df_final["annotator_notes"] = ""   # catatan anotator

df_final = df_final.sort_values(
    by="created_at", ascending=True).reset_index(drop=True)
is_sorted = df_final["created_at"].is_monotonic_increasing
print(f"Verifikasi urutan ascending: {'SORTED' if is_sorted else 'NOT SORTED'}")

df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"\n=== OUTPUT ===")
print(f"Path   : {OUTPUT_PATH}")
print(f"Shape  : {df_final.shape[0]:,} baris x {df_final.shape[1]} kolom")
print(f"Kolom  : {list(df_final.columns)}")
print(f"\nPreview 3 baris pertama:")
display(df_final[["id_str","created_at","period","full_text",
                  "word_count","source_keyword","label"]].head(3))
print(f"\nPreview 3 baris terakhir:")
display(df_final[["id_str","created_at","period","full_text",
                  "word_count","source_keyword","label"]].tail(3))


## 9. EDA: Distribusi Panjang Token

Distribusi panjang teks (proxy word count) dengan garis batas `max_len = 128` sebagai referensi tokenisasi IndoBERT.


In [ ]:
MAX_LEN     = 128
token_len   = df_final["word_count"].dropna().astype(int)
n_total     = len(token_len)
n_within    = (token_len <= MAX_LEN).sum()
n_truncated = (token_len >  MAX_LEN).sum()
pct_within  = n_within    / n_total * 100
pct_trunc   = n_truncated / n_total * 100
mean_val    = token_len.mean()
median_val  = token_len.median()

fig, ax = plt.subplots(figsize=(13, 5))
n_bins = min(60, int(token_len.max() - token_len.min()))
ax.hist(token_len, bins=n_bins, color="#534AB7", alpha=0.7,
        edgecolor="none", density=False, label="Frekuensi")
ax2 = ax.twinx()
token_len.plot.kde(ax=ax2, color="#E24B4A", linewidth=2, label="Densitas (KDE)")
ax2.set_ylabel("Densitas", fontsize=10, color="#E24B4A")
ax2.tick_params(axis="y", colors="#E24B4A")
ax2.set_ylim(bottom=0)
ax.axvline(x=MAX_LEN, color="#EF9F27", linestyle="--", linewidth=2,
           label=f"max_len = {MAX_LEN}")
ax.axvline(x=mean_val, color="#1D9E75", linestyle=":", linewidth=1.5,
           label=f"Mean = {mean_val:.1f}")
ax.axvline(x=median_val, color="#888780", linestyle=":", linewidth=1.5,
           label=f"Median = {median_val:.0f}")
ax.text(MAX_LEN + 1.5, ax.get_ylim()[1] * 0.88,
        f"<={MAX_LEN} kata: {n_within:,} ({pct_within:.1f}%)\n"
        f">{MAX_LEN} kata: {n_truncated:,} ({pct_trunc:.1f}%)",
        fontsize=9, color="#444441",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#FFF8E1",
                  edgecolor="#EF9F27", alpha=0.9))
ax.set_xlabel("Panjang Tweet (jumlah kata, proxy word count)", fontsize=11)
ax.set_ylabel("Frekuensi", fontsize=11)
ax.set_title(
    f"Distribusi Panjang Token Dataset Final (Pra-Preprocessing)\n"
    f"N = {n_total:,} tweet  |  12 periode  |  6 Jan 2025 - 6 Jan 2026",
    fontsize=12, pad=12)
lines1, lbl1 = ax.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, lbl1+lbl2, fontsize=9, loc="upper right")
sns.despine(right=False); plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/distribusi_panjang_token.png",
            dpi=150, bbox_inches="tight")
plt.show()

print("=== STATISTIK PANJANG TOKEN ===")
print(f"Total tweet                : {n_total:,}")
print(f"Mean                       : {mean_val:.2f} kata")
print(f"Median                     : {median_val:.0f} kata")
print(f"Min                        : {token_len.min()} kata")
print(f"Max                        : {token_len.max()} kata")
print(f"Dalam batas <={MAX_LEN}    : {n_within:,} ({pct_within:.2f}%)")
print(f"Melebihi batas >{MAX_LEN}  : {n_truncated:,} ({pct_trunc:.2f}%)")


## 10. Summary

In [ ]:
print("=" * 65)
print(" DATA PREPARATION SUMMARY")
print("=" * 65)
print(f" Output dir            : {WORK_DIR}")
print(f" {'-'*62}")
print(f" Pipeline per keyword:")
print(f"   Makan Bergizi Gratis  : {len(df_makan):>8,} tweet")
print(f"   MBG                   : {len(df_mbg):>8,} tweet")
print(f" Merged full             : {len(df_full):>8,} tweet")
print(f" {'-'*62}")
print(f" Pre-filter:")
print(f"   Masuk pre-filter      : {n_pf0:>8,} tweet")
print(f"   Setelah filter null   : {n_pf1:>8,} tweet")
print(f"   Setelah filter panjang: {n_pf2:>8,} tweet")
print(f"   Setelah assign_period : {len(df):>8,} tweet")
print(f" {'-'*62}")
print(f" Stratified Sampling:")
print(f"   Target/periode        : {SAMPLE_PER_PERIOD:>8,}")
print(f"   Periode normal        : {(~df_log['is_low_volume']).sum():>8} periode")
print(f"   Periode low-volume    : {df_log['is_low_volume'].sum():>8} periode")
print(f" {'-'*62}")
print(f" OUTPUT FINAL            : {len(df_final):>8,} tweet")
print(f" Random seed             : {RANDOM_SEED}")
print(f" Output path             : {OUTPUT_PATH}")
print("=" * 65)
print("\nLangkah selanjutnya (Notebook 02):")
print("  1. Rename raw_sampling_mbg.csv -> raw_sample_labeled_mbg.csv")
print("  2. Anotasi manual kolom 'label' (positive/negative/neutral)")
print("  3. Jalankan NB02 untuk preprocessing dan validasi via IndoRoBERTa")
